In [21]:
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from tqdm.notebook import tqdm
import pickle

In [13]:
df = pd.read_csv("galaxyClusters.csv")
ra_col, dec_col, obsid_col = "RA", "Dec", "Obs ID"

coords = SkyCoord(
    ra=df[ra_col].astype(str).str.strip().values,
    dec=df[dec_col].astype(str).str.strip().values,
    unit=(u.hourangle, u.deg),
    frame="icrs",
)

df['RA_calc'] = [coord.ra.deg for coord in coords]
df['Dec_calc'] = [coord.dec.deg for coord in coords]

In [14]:
df

,Seq Num,Obs ID,Instrument,Grating,Appr Exp,Exposure,Target Name,PI Name,RA,Dec,...,Public Release Date,Proposal,Science Category,Type,Obs Cycle,Prop Cycle,Joint,Grid Name,RA_calc,Dec_calc
0,600079,319,ACIS-S,NONE,60.00,56.04,NGC 1399,Mushotzky,03 38 29.40,-35 27 00.40,...,2001-02-09 15:00:00,1600145,CLUSTERS OF GALAXIES,GTO,1,1,NaN,NaN,54.622500,-35.450111
1,600080,239,ACIS-I,NONE,3.78,3.60,NGC 1399,Mushotzky,03 38 29.40,-35 27 00.40,...,2001-05-25 14:05:00,1600145,CLUSTERS OF GALAXIES,GTO,1,1,NaN,NaN,54.622500,-35.450111
2,600080,320,ACIS-I,NONE,3.50,3.38,NGC 1399,Mushotzky,03 38 29.40,-35 27 00.40,...,2001-05-25 14:05:00,1600145,CLUSTERS OF GALAXIES,GTO,1,1,NaN,NaN,54.622500,-35.450111
3,600081,321,ACIS-S,NONE,40.00,39.59,NGC 4472,Mushotzky,12 29 46.90,+08 00 13.00,...,2001-07-18 18:30:00,1600145,CLUSTERS OF GALAXIES,GTO,1,1,NaN,NaN,187.445417,8.003611
4,600082,322,ACIS-I,NONE,10.00,10.36,NGC 4472,Mushotzky,12 29 46.90,+08 00 13.00,...,2001-04-07 17:20:00,1600145,CLUSTERS OF GALAXIES,GTO,1,1,NaN,NaN,187.445417,8.003611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3744,890259,29629,ACIS-S,NONE,10.00,9.46,Perseus Cluster,Calibration,03 19 48.20,+41 30 42.10,...,2025-09-19 21:30:34,26800469,CLUSTERS OF GALAXIES,CAL,26,26,NaN,NaN,49.950833,41.511694
3745,890260,29630,ACIS-S,NONE,10.00,9.95,Perseus Cluster,Calibration,03 19 48.20,+41 30 42.10,...,2025-09-21 09:00:22,26800469,CLUSTERS OF GALAXIES,CAL,26,26,NaN,NaN,49.950833,41.511694
3746,890261,31035,ACIS-I,NONE,25.00,22.78,A1795,Calibration,13 48 52.70,+26 35 27.00,...,2025-08-21 20:04:02,26800462,CLUSTERS OF GALAXIES,CAL,26,26,NaN,NaN,207.219583,26.590833
3747,890272,31411,ACIS-I,NONE,20.00,19.83,A1795,Calibration,13 48 52.70,+26 35 27.00,...,2026-01-16 10:47:03,27800384,CLUSTERS OF GALAXIES,CAL,27,27,NaN,NaN,207.219583,26.590833


In [15]:
max_sep = 10.0 * u.arcsec


In [19]:
import astropy.units as u
from astropy.coordinates import SkyCoord


clusters = []  # list of dicts: {"center": SkyCoord, "obsids": [..], "members": [row indices]}
# or if you only want obsids: {"center": SkyCoord, "obsids": [...]}

for idx, row in tqdm(df.iterrows()):
    # Make a SkyCoord for this entry (RA_calc/Dec_calc are in degrees)
    c = SkyCoord(row["RA_calc"] * u.deg, row["Dec_calc"] * u.deg, frame="icrs")

    # Try to match to an existing cluster center
    matched = False
    for cl in clusters:
        if c.separation(cl["center"]) <= max_sep:
            cl["obsids"].append(int(row["Obs ID"]))
            cl["members"].append(idx)
            matched = True
            break

    # If no match, start a new cluster
    if not matched:
        clusters.append(
            {
                "center": c,
                "obsids": [int(row["Obs ID"])],
                "members": [idx],
            }
        )

print("Num clusters:", len(clusters))


0it [00:00, ?it/s]

Num clusters: 1969


In [22]:
with open('./clusters_grouped.pkl', 'wb') as f:
    pickle.dump(clusters, f)